以下を行う生成モデルを作成する。
```
Gaussian distribution
        ↓
   Generative Model
        ↓
簡単な分布

```



## 時刻の対応

通常の diffusion では

$$
k=0:\mathrm{data},
\qquad
k=T:\mathrm{noise}
$$

です。

Flow Matching と向きを合わせるため、

$$
t=0:\mathrm{noise},
\qquad
t=1:\mathrm{data}
$$

としたい。

そこで

$$
k=T(1-t)
$$

と置く。

---

## forward process を (t) で書く

通常の forward diffusion は data から noise へ進むので、
Flow Matching 向きの (t) では **(t=1) から (t=0) へ進む**。

$$
x_{t=1}^{\mathrm{data}}
\rightarrow
x_{t=1-\Delta t}
\rightarrow
\cdots
\rightarrow
x_{t=\Delta t}
\rightarrow
x_{t=0}^{\mathrm{noise}}
$$

式は

$$
x_{T(1-t)}
=
\sqrt{\bar{\alpha}_{T(1-t)}}x_{\mathrm{data}}
+
\sqrt{1-\bar{\alpha}_{T(1-t)}}\epsilon
$$

である。



---

## backward process を (t) で書く

生成過程である backward diffusion は noise から data へ進むので、
Flow Matching と同じ向きで

$$
x_{t=0}^{\mathrm{noise}}
\rightarrow
x_{t=\Delta t}
\rightarrow
\cdots
\rightarrow
x_{t=1-\Delta t}
\rightarrow
x_{t=1}^{\mathrm{data}}
$$

と書ける。

DDPM では各ステップで

$$
p_\theta(x_{t+\Delta t}\mid x_t)
$$

を用いて更新する。

すなわち、

$$
x_{t+\Delta t}
\sim
p_\theta(x_{t+\Delta t}\mid x_t)
$$

である。

通常の diffusion 添字 (k=T(1-t)) を使うと、これは

$$
p_\theta(x_{k-1}\mid x_k)
$$

に対応する。

---

## まとめ

Flow Matching と同じ向きで書くと、

### forward diffusion

$$
x_{t=1}^{\mathrm{data}}
\rightarrow
x_{t=0}^{\mathrm{noise}}
$$

### backward diffusion

$$
x_{t=0}^{\mathrm{noise}}
\rightarrow
x_{t=1}^{\mathrm{data}}
$$

です。

つまり、**forward diffusion は (t) に対して逆向き、backward diffusion は (t) に対して順向き**です。


# backward diffusion

生成時には、まずランダム分布

$$
x_{t=0}
\sim
\mathcal N(0,I)
$$

から開始する。

各時刻において neural network は

$$
\epsilon_\theta(x_t,t)
$$

を用いて、現在のサンプルに含まれる Gaussian ノイズを推定する。

DDPM の逆過程は

$$
p_\theta(x_{t+\Delta t}\mid x_t)
$$

を逐次適用することで実現される。

具体的には、推定したノイズを用いて現在のサンプルを少しずつ denoise し、

$$
x_{t=0}
\rightarrow
x_{\Delta t}
\rightarrow
\cdots
\rightarrow
x_{1-\Delta t}
\rightarrow
x_{t=1}
$$

と更新する。

この過程により、単純な Gaussian 分布は徐々にデータ分布へ変換される。

すなわち、

$$
\mathcal N(0,I)
\rightarrow
p_{\mathrm{data}}(x)
$$

という分布変換を実現している。

Flow Matching と比較すると、

$$
x_{t=0}^{\mathrm{noise}}
\rightarrow
x_{t=1}^{\mathrm{data}}
$$

という生成方向は同じであり、両者とも

「単純な分布から複雑なデータ分布への輸送」

を実現している。

ただし、Flow Matching が速度場

$$
u_\theta(x_t,t)
$$

を学習して ODE を積分するのに対し、DDPM はノイズ

$$
\epsilon_\theta(x_t,t)
$$

を学習し、確率的な逆拡散過程を逐次適用する。


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# -----------------------
# setup
# -----------------------
device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(0)
np.random.seed(0)

# -----------------------
# target distribution
# 1D bimodal Gaussian mixture
# -----------------------
def sample_target(n):

    comp = np.random.rand(n) > 0.55

    mu = np.where(
        comp,
        1.5,
        -0.85
    )

    sigma = np.where(
        comp,
        0.25,
        0.65
    )

    x = np.random.normal(
        mu,
        sigma
    )

    return (
        torch.tensor(
            x,
            dtype=torch.float32
        )
        .view(-1, 1)
        .to(device)
    )

# -----------------------
# diffusion schedule
# beta scheduling
# -----------------------
T = 100

betas = torch.linspace(
    1e-4,
    0.02,
    T,
    device=device
)

alphas = 1.0 - betas

alpha_bars = torch.cumprod(
    alphas,
    dim=0
)

# -----------------------
# noise prediction model
#
# eps_theta(x_k, k)
# -----------------------
class DiffusionModel(nn.Module):

    def __init__(self, hidden=64):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, hidden),
            nn.GELU(),

            nn.Linear(hidden, hidden),
            nn.GELU(),

            nn.Linear(hidden, 1),
        )

    def forward(self, x_k, k):

        k = (
            k.float()
            .view(-1, 1)
            / T
        )

        return self.net(
            torch.cat(
                [x_k, k],
                dim=1
            )
        )

model = DiffusionModel().to(device)

![fig/1515_forward_model.png](fig/1515_forward_model.png)

In [ ]:
opt = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)


def q_sample(x0, k, eps):

    alpha_bar_k = (
        alpha_bars[k]
        .view(-1, 1)
    )

    x_k = (
        torch.sqrt(alpha_bar_k) * x0
        +
        torch.sqrt(
            1.0 - alpha_bar_k
        ) * eps
    )

    return x_k

この関数は **forward diffusion**

$$
q(x_k|x_0)
$$

を実装しています。

数式

$$
x_k
=
\sqrt{\bar{\alpha}_k},x_0
+
\sqrt{1-\bar{\alpha}_k},\epsilon
$$



In [ ]:
# -----------------------
# training
# -----------------------
batch_size = 256
n_iter = 5000

losses = []

for step in range(n_iter):
    # -------------------
    # sample data x_0
    # -------------------
    x0 = sample_target(batch_size)

    # -------------------
    # sample timestep k
    # -------------------
    k = torch.randint(0, T, (batch_size,), device=device)

    # -------------------
    # Gaussian random
    # eps ~ N(0,I)
    # -------------------
    eps = torch.randn_like(x0)

    # -------------------
    # create noisy sample
    #
    # x_k
    # =
    # sqrt(alpha_bar_k)x_0
    # +
    # sqrt(1-alpha_bar_k)eps
    # -------------------
    x_k = q_sample(x0, k, eps)

    # -------------------
    # predict eps
    # -------------------
    eps_pred = model(x_k, k)

    loss = ((eps_pred - eps) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

    losses.append(loss.item())

    if step % 500 == 0:
        print(step, loss.item())

![fig/1515_training.png](fig/1515_training.png)

In [ ]:
# -----------------------
# reverse diffusion
#
# x_T -> ... -> x_0
# -----------------------
@torch.no_grad()
def sample(model, n=10000):
    model.eval()

    # -------------------
    # start from noise
    #
    # x_T ~ N(0,I)
    # -------------------
    x = torch.randn(n, 1, device=device)

    for k in reversed(range(T)):
        k_batch = torch.full((n,), k, device=device, dtype=torch.long)

        beta_k = betas[k]
        alpha_k = alphas[k]
        alpha_bar_k = alpha_bars[k]

        # -------------------
        # predict noise
        #
        # eps_theta(x_k,k)
        # -------------------
        eps_pred = model(x, k_batch)

        # -------------------
        # DDPM mean
        # -------------------
        mean = (
            1.0
            / torch.sqrt(alpha_k)
            * (x - beta_k / torch.sqrt(1.0 - alpha_bar_k) * eps_pred)
        )

        if k > 0:
            eps = torch.randn_like(x)
            x = mean + torch.sqrt(beta_k) * eps
        else:
            x = mean

    return x.cpu().numpy().ravel()

![fig/1515_generation.png](fig/1515_generation.png)

In [ ]:
# -----------------------
# generate samples
# -----------------------
x_gen = sample(model, n=10000)

# -----------------------
# plot
# -----------------------
plt.figure(figsize=(8, 3))
plt.hist(x_gen, bins=100, density=True, alpha=0.6, label="generated")
x_true = sample_target(10000).cpu().numpy().ravel()
plt.hist(x_true, bins=100, density=True, alpha=0.4, label="target")
plt.legend()
plt.xlabel("x")
plt.ylabel("density")
plt.title("1D DDPM Diffusion Model")
plt.tight_layout()
plt.show()

# -----------------------
# training loss
# -----------------------
plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel("iteration")
plt.ylabel("loss")
plt.title("training loss")
plt.tight_layout()
plt.show()